# EGM Veri Toplama — BES Kalıcılık ve Rekabet Projesi

Emeklilik Gözetim Merkezi'nin (egm.org.tr) yayınladığı BES istatistik raporlarını
tarih tarih indirir. Her rapor tek bir tarihin anlık görüntüsüdür; zaman serisi
için tarih listesindeki her tarih tek tek geziliyor.

**Hedef raporlar**

| Kod | Rapor | Ne için |
|---|---|---|
| `GR-01-015` | Sözleşmelerin Yürürlüğe Giriş Ayına Göre Sonlanma Oranı | Kohort kalıcılık eğrisi (2003.10 →) |
| `GR-01-010` | Sonlanan Sözleşmelerin Sonlanma Nedenine Göre Dağılımı | Ayrılma nedenleri kırılımı |
| `GR-01-019` | Cayma Hakkı Kullanım Oranı | Cayma trendi |
| `EGM-05-006` | Emeklilik Şirketi ve Faiz Detayında Fon Büyüklükleri | Şirket bazında pazar payı |
| `EGM-05-001` | Fon Bazında Fon İşletim Gideri Kesinti Oranları | Maliyet rekabeti |

**Akış:** Kurulum → Script → Keşif (`--probe`) → Küçük deneme → Tam toplama → İndir

---

> **Kaynak:** Bu defterde toplanan tüm istatistiksel veriler Emeklilik Gözetim
> Merkezi A.Ş.'ye (EGM) aittir. Kaynak gösterilmeden alıntı yapılamaz.

## 1. Kurulum

Playwright ve Chromium kuruluyor. EGM sayfası JavaScript ile render edildiği için
`requests` boş tablo döndürür; gerçek bir tarayıcı motoru gerekiyor.

Colab oturumu sıfırlandığında bu hücre tekrar çalıştırılmalı.

In [ ]:
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium

## 2. Toplayıcı script'i

Script repo'da (`scripts/egm_collector.py`) duruyor; buraya kopyalanmak yerine
oradan çekiliyor. Tek doğru nüsha repo olduğu için defterdeki kopya eskimiyor.

Ayrı bir dosya olarak çalıştırılmasının nedeni: Playwright'ın sync API'si
notebook'un asyncio döngüsüyle çakışıyor. `!python` ile ayrı süreç olarak
çalıştırıldığında sorun kalmıyor.

In [ ]:
!wget -q -O egm_collector.py https://raw.githubusercontent.com/esero88/eser-portfolio/main/pension-retention-analytics/scripts/egm_collector.py
!wc -l egm_collector.py

## 3. Keşif

`--probe` hiçbir şey indirmez; sayfayı açıp iki şeyi raporlar:

- Tarih listesi bulundu mu, kaç tarih var
- "Excel'e Aktar" düğmesi görülebiliyor mu

`RAPOR_URL` değişkenine raporun EGM'deki adresi yazılıyor.

In [ ]:
RAPOR_URL = ""   # <-- raporun linkini buraya yapıştır
RAPOR_KODU = "GR-01-015"

assert RAPOR_URL, "Önce RAPOR_URL'i doldur"
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --probe

### Çıktının okunması

| Çıktı | Anlamı |
|---|---|
| "N tarih bulundu" + "Excel düğmesi: BULUNDU" | Yolunda, toplamaya geçilebilir |
| "tarih listesi bulunamadı" | Tarih seçici normal bir `<select>` değil; `tarih_secicisini_bul()` güncellenmeli |
| "Excel düğmesi: BULUNAMADI" | `export_dugmesini_bul()` güncellenmeli |
| Zaman aşımı / bağlantı hatası | EGM, Colab'ın IP'sini engelliyor olabilir; script yerel makinede çalıştırılmalı |

## 4. Küçük deneme

Tam toplamadan önce üç tarihle sınanıyor: dosyalar iniyor mu, içerikleri dolu mu.

In [ ]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out data/raw --limit 3

In [ ]:
import pandas as pd
from pathlib import Path

dosyalar = sorted(Path(f"data/raw/{RAPOR_KODU}").glob("*.xlsx"))
print(f"{len(dosyalar)} dosya indi:")
for d in dosyalar:
    print(" ", d.name, f"({d.stat().st_size:,} byte)")

if dosyalar:
    ornek = pd.read_excel(dosyalar[0])
    print(f"\n{dosyalar[0].name}  ->  {ornek.shape[0]} satır × {ornek.shape[1]} kolon")
    display(ornek.head(10))

## 5. Tam toplama

Tarih listesinin tamamı indiriliyor. Zaten inmiş dosyalar atlanıyor, yarıda
kesilirse tekrar çalıştırıldığında kaldığı yerden devam ediyor.

`--delay 2.5` istekler arasındaki bekleme; siteyi yormamak için bırakılmıştır.

In [ ]:
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --out data/raw --delay 2.5

## 6. Diğer raporlar

Her raporun kendi adresi var. Aynı işlem, rapor kodu ve link değiştirilerek
tekrarlanıyor.

In [ ]:
# Örnek: şirket bazında fon büyüklükleri
RAPOR_URL_2 = ""   # <-- EGM-05-006'nın linki
RAPOR_KODU_2 = "EGM-05-006"

if RAPOR_URL_2:
    !python egm_collector.py --url "$RAPOR_URL_2" --code "$RAPOR_KODU_2" --out data/raw --delay 2.5

## 7. Toplananları indir

Colab oturumu kapandığında dosyalar siliniyor. İki seçenek: zip olarak indirmek
ya da Drive'a kopyalamak.

In [ ]:
# Seçenek A — zip'leyip bilgisayarına indir
from google.colab import files
!zip -rq egm_data.zip data/raw
files.download("egm_data.zip")

In [ ]:
# Seçenek B — Google Drive'a kaydet
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/egm-bes-projesi"
# !cp -r data/raw "/content/drive/MyDrive/egm-bes-projesi/"

---

## Sonraki adım

Ham dosyalar toplandıktan sonra:

1. **Temizlik** — iki satıra bölünmüş başlıkların düzleştirilmesi, bilimsel
   gösterimdeki sayıların düzeltilmesi, uzun formata (`rapor_tarihi, şirket,
   segment, metrik, değer`) çevrilmesi.
2. **Ambar** — temiz tabloların PostgreSQL'e yüklenmesi, yıldız şema.
3. **Dashboard** — Tableau: kohort kalıcılık eğrisi, sonlanma nedenleri,
   pazar payı, maliyet karşılaştırması.
4. **Agent** — metrik sözlüğü, doğal dilden SQL, 30 soruluk doğruluk testi.